### Configure PyTorch CUDA memory allocation settings to prevent fragmentation

In [1]:
 import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

### Check GPU availability and display GPU information

In [2]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)

Tue Mar 17 16:49:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [3]:
 %%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate

### Install system dependencies and clone the KenLM language model toolkit

In [4]:
!sudo apt-get update
!sudo apt-get install build-essential libboost-all-dev cmake zlib1g-dev libbz2-dev liblzma-dev
!sudo apt-get install libboost-all-dev libeigen3-dev
!git clone https://github.com/kpu/kenlm
%cd kenlm
!pip install e .

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,442 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

### Build and install KenLM from source

In [5]:
!mkdir build
%cd build
!cmake ..
!make -j 4
!sudo make install

mkdir: cannot create directory ‘build’: File exists
/content/kenlm/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.74.0/BoostConfig.cmake (found suitable v

### Add KenLM binaries to the system PATH and verify installation

In [6]:
import os
os.environ['PATH'] += ":/content/kenlm/build/bin"  # Use your exact path
!echo $PATH  # Verify
!which lmplz  # Test

/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/kenlm/build/bin
/usr/local/bin/lmplz


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [7]:
import re
import json
import unicodedata
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

from datasets import load_dataset, Audio, concatenate_datasets
import evaluate

from transformers import (
    Wav2Vec2CTCTokenizer,
    SeamlessM4TFeatureExtractor,
    Wav2Vec2BertProcessor,
    Wav2Vec2BertForCTC,
    TrainingArguments,
    Trainer,
)

### Load the Dhivehi Common Voice dataset and remove unused metadata columns

In [8]:
dv_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
dv_test  = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

dv_train = dv_train.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])
dv_test  = dv_test.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 119565.13it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 106488.52it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 118258.30it/s]


### Install the wget package for downloading files

In [9]:
!pip install wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=d2143270f6df6d55c3c303f52eeab5dc9b2f3fead31dc08dc9beaec0b8fd91f2
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


### Load the Sinhala ASR dataset with specified train/test splits

In [11]:
import os
import aiohttp
from datasets import load_dataset

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "180"

splits = load_dataset(
    "keshan/large-sinhala-asr-dataset",
    "si",
    split={
        "train": "train[:40%]",
        "test": "test[:40%]",
    },
    trust_remote_code=True,
    storage_options={
        "client_kwargs": {
            "timeout": aiohttp.ClientTimeout(total=3600)
        }
    },
)

si_train = splits["train"].rename_column("file", "audio").remove_columns(["filename", "x"])
si_test  = splits["test"].rename_column("file", "audio").remove_columns(["filename", "x"])


print("Sinhala train size:", len(si_train))
print("Sinhala test size:", len(si_test))

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Sinhala train size: 53018
Sinhala test size: 9356


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [12]:
 # -------- Dhivehi cleaning --------
chars_to_remove_regex_dv = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«]'
arabic_pattern = re.compile(r"[\u0600-\u06FF]")

def dv_clean(batch):
    s = re.sub(chars_to_remove_regex_dv, "", batch["sentence"]).lower()
    # remove Arabic rows
    if arabic_pattern.search(s):
        batch["sentence"] = ""
        return batch
    # remove latin
    s = re.sub(r"[a-z]+", "", s)
    batch["sentence"] = s.strip()
    return batch

# -------- Sinhala cleaning --------
chars_to_remove_regex_si = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\(\)\'\”\‘‘\’\ʻ\”\“\–\—\…\»\«।॥]'
sinhala_allowed = re.compile(r"^[\u0D80-\u0DFF\s]+$")

def si_clean(batch):
    s = unicodedata.normalize("NFC", batch["sentence"])
    s = re.sub(chars_to_remove_regex_si, "", s)
    s = re.sub(r"[A-Za-z0-9]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    batch["sentence"] = s
    return batch

### Apply cleaning functions and filter out empty sentences from both Dhivehi and Sinhala datasets

In [13]:
dv_train = dv_train.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)
dv_test  = dv_test.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)

si_train = si_train.map(si_clean).filter(lambda x: bool(sinhala_allowed.match(x["sentence"])) if x["sentence"] else False)
si_test  = si_test.map(si_clean).filter(lambda x: bool(sinhala_allowed.match(x["sentence"])) if x["sentence"] else False)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

Map:   0%|          | 0/53018 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53018 [00:00<?, ? examples/s]

Map:   0%|          | 0/9356 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9356 [00:00<?, ? examples/s]

### Resample all audio to 16kHz for both Dhivehi and Sinhala datasets

In [14]:
dv_train = dv_train.cast_column("audio", Audio(sampling_rate=16_000))
dv_test  = dv_test.cast_column("audio", Audio(sampling_rate=16_000))
si_train = si_train.cast_column("audio", Audio(sampling_rate=16_000))
si_test  = si_test.cast_column("audio", Audio(sampling_rate=16_000))

### Extract the combined character vocabulary from both Dhivehi and Sinhala datasets

In [15]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

all_train_text = concatenate_datasets([
    dv_train.select_columns(["sentence"]),
    si_train.select_columns(["sentence"]),
])
all_test_text = concatenate_datasets([
    dv_test.select_columns(["sentence"]),
    si_test.select_columns(["sentence"]),
])

vocab_train = all_train_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_train_text.column_names
)
vocab_test = all_test_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_test_text.column_names
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

# Space -> word delimiter |
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]


# Specials
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

print("Vocab size:", len(vocab_dict))

Map:   0%|          | 0/47475 [00:00<?, ? examples/s]

Map:   0%|          | 0/9687 [00:00<?, ? examples/s]

Vocab size: 131


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [16]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

### Write all transcriptions to a text file for KenLM language model training

In [17]:
with open("lm_corpus.txt", "w", encoding="utf-8") as f:
    for ex in all_train_text:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")
    for ex in all_test_text:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")

print("Wrote LM corpus to lm_corpus.txt")

Wrote LM corpus to lm_corpus.txt


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [18]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

dv_train_p = dv_train.map(prepare_dataset, remove_columns=dv_train.column_names)
dv_test_p  = dv_test.map(prepare_dataset,  remove_columns=dv_test.column_names)
si_train_p = si_train.map(prepare_dataset, remove_columns=si_train.column_names)
si_test_p  = si_test.map(prepare_dataset,  remove_columns=si_test.column_names)

Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

### Concatenate Dhivehi and Sinhala processed datasets into unified train and test sets

In [19]:
train = concatenate_datasets([dv_train_p, si_train_p]).shuffle(seed=42)
test  = concatenate_datasets([dv_test_p,  si_test_p]).shuffle(seed=42)

print("Train size:", len(train))
print("Test size:", len(test))


Train size: 47475
Test size: 9687


### Define a custom data collator that pads input features and labels for CTC training

In [20]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

### Train a trigram KenLM language model from the corpus

In [21]:
!lmplz -o 3 < lm_corpus.txt > lm.arpa

=== 1/5 Counting and sorting n-grams ===
Reading /content/kenlm/build/lm_corpus.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 260747 types 52033
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:624396 2:15822624768 3:29667422208
Statistics:
1 52033 D1=0.713309 D2=1.08887 D3+=1.4221
2 174425 D1=0.874974 D2=1.15591 D3+=1.44164
3 203937 D1=0.73259 D2=1.57853 D3+=2.26804
Memory estimate for binary LM:
type       kB
probing  8994 assuming -p 1.5
probing 10219 assuming -r models -p 1.5
trie     4454 without quantization
trie     2884 assuming -q 8 -b 8 quantization 
trie     4224 assuming -a 22 array pointer compression
trie     2654 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:624396 2:2790800 3:4078740
---

### Install pyctcdecode library for CTC beam search decoding with language model support

In [22]:
!pip install pyctcdecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.3/529.3 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 131.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
raster

### Build a CTC beam search decoder with KenLM language model integration

In [23]:
from pyctcdecode import build_ctcdecoder

# get labels in ID order
vocab = processor.tokenizer.get_vocab()  # token -> id
id2token = {v: k for k, v in vocab.items()}
labels = [id2token[i] for i in range(len(id2token))]

# path to your trained KenLM model (arpa or binary)
kenlm_model_path = "lm.arpa"  # <-- change if you use another path/name

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_model_path,
    alpha=0.5,  # tune on dev
    beta=1.0  # tune on dev
)


### Define the evaluation metrics function using LM-aided beam search decoding to compute WER and CER

In [24]:
import evaluate
import torch

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    # Convert logits to probabilities
    probs = torch.softmax(torch.tensor(pred_logits), dim=-1).numpy()

    # Decode with LM
    pred_str = [decoder.decode(p, beam_width=64) for p in probs]

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Mount Google Drive for persistent storage

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Run greedy CTC inference on the test set and compute WER/CER

In [26]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

all_pred = []
all_refs = []

checkpoint_dir = "/content/drive/MyDrive/final_model40"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

model.eval()

for ex in dv_test_p:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    # Convert logits to probabilities for LM decoding
    probs = torch.softmax(logits[0], dim=-1).cpu().numpy()

    # Decode with language model
    pred_text = decoder.decode(probs)

    # Reference text
    ref_text = processor.decode(ex["labels"], group_tokens=False)

    all_pred.append(pred_text)
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (Dhivehi only, LM decode): {wer:.4f}")
print(f"CER (Dhivehi only, LM decode): {cer:.4f}")

for idx in range(min(3, len(all_refs))):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (Dhivehi only, LM decode): 0.1326
CER (Dhivehi only, LM decode): 0.0308

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
PRED: ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
PRED: އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
PRED: އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ


### Run greedy CTC inference on the test set and compute WER/CER

In [27]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

all_pred = []
all_refs = []

checkpoint_dir = "/content/drive/MyDrive/final_model40"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

model.eval()

for ex in si_test_p:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    # Convert logits to probabilities for LM decoding
    probs = torch.softmax(logits[0], dim=-1).cpu().numpy()

    # Decode with language model
    pred_text = decoder.decode(probs)

    # Reference text
    ref_text = processor.decode(ex["labels"], group_tokens=False)

    all_pred.append(pred_text)
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (Sinhala only, LM decode): {wer:.4f}")
print(f"CER (Sinhala only, LM decode): {cer:.4f}")

for idx in range(min(3, len(all_refs))):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (Sinhala only, LM decode): 0.0649
CER (Sinhala only, LM decode): 0.0178

--- Example 0 ---
REF : ආචාර්යවරිය නම් ලකුණු කරන අතරතුර
PRED: ආචාර්යවරිය නම් ලකුණු කරන අතරතුර

--- Example 1 ---
REF : මේ අයුරින් මරණයට පත් වේ
PRED: මේ අයුරින් මරණයට පත් වේ

--- Example 2 ---
REF : ඒකට මොකද මේකෙන්
PRED: ඒකට මොකද මේකෙන්
